In [3]:
def debug_rst_tree_processing():
    """RST 트리 처리 문제 디버깅"""
    import traceback
    import os
    import time
    import sys
    
    # 프로젝트 루트 경로를 Python 경로에 추가
    current_dir = os.getcwd()
    if "src" not in sys.path:
        sys.path.append(os.path.join(current_dir, "src"))
    
    # 로그 파일 설정
    log_file = f"rst_debug_{time.strftime('%Y%m%d_%H%M%S')}.log"
    
    def log(msg):
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(f"[{time.strftime('%H:%M:%S')}] {msg}\n")
    
    log("디버깅 시작")
    log(f"현재 작업 디렉토리: {current_dir}")
    log(f"Python 경로: {sys.path}")
    
    try:
        # 문제가 되는 문서 ID 하드코딩
        doc_id = "wsj_1100"
        
        # 데이터 로드
        import json
        log("데이터 로드 중...")
        raw_data = json.load(open("data/rstdt/train.json"))
        doc = next((d for d in raw_data if d["doc_id"] == doc_id), None)
        
        if not doc:
            log(f"문서 {doc_id}를 찾을 수 없습니다")
            return
        
        log(f"문서 {doc_id} 로드 완료")
        
        # RST 트리 문자열 저장
        log("RST 트리 문자열 저장 중...")
        with open(f"{doc_id}_rst_string.txt", "w", encoding="utf-8") as f:
            f.write(doc["rst_tree"])
        
        # 트리 파싱 - 직접 경로 지정
        log("모듈 임포트 시도...")
        try:
            from src.data.tree import RSTTree
            log("src.data.tree에서 RSTTree 임포트 성공")
        except ImportError:
            try:
                from data.tree import RSTTree
                log("data.tree에서 RSTTree 임포트 성공")
            except ImportError:
                log("RSTTree 임포트 실패, 절대 경로 시도...")
                # 프로젝트 구조에 따라 경로 조정 필요
                module_path = os.path.abspath(os.path.join(current_dir))
                if module_path not in sys.path:
                    sys.path.insert(0, module_path)
                    log(f"경로 추가: {module_path}")
                
                try:
                    from src.data.tree import RSTTree
                    log("절대 경로로 RSTTree 임포트 성공")
                except ImportError as e:
                    log(f"모든 임포트 시도 실패: {e}")
                    return
        
        log("RST 트리 파싱 중...")
        rst_tree = RSTTree.fromstring(doc["rst_tree"])
        log("트리 파싱 완료")
        
        # 트리 기본 정보 저장
        positions = list(rst_tree.treepositions())
        log(f"트리 위치 수: {len(positions)}")
        log(f"처음 10개 위치: {positions[:10]}")
        
        # 위치 (0,1) 문제 확인을 위해 선행 위치들 체크
        for pos in [(), (0,), (0,0), (0,1)]:
            log(f"\n위치 {pos} 체크 중...")
            
            try:
                if pos in positions:
                    log(f"위치 {pos}가 트리에 존재함")
                    # 스레드 기반 타임아웃으로 안전하게 접근
                    import threading
                    result = [None]
                    exception = [None]
                    finished = [False]
                    
                    def access_node():
                        try:
                            result[0] = rst_tree[pos]
                        except Exception as e:
                            exception[0] = e
                        finally:
                            finished[0] = True
                    
                    thread = threading.Thread(target=access_node)
                    thread.daemon = True
                    
                    log(f"위치 {pos} 접근 시도 (타임아웃 5초)...")
                    thread.start()
                    thread.join(5)  # 5초 타임아웃
                    
                    if not finished[0]:
                        log(f"위치 {pos} 접근 시간 초과!")
                        continue
                    
                    if exception[0]:
                        log(f"위치 {pos} 접근 중 예외: {exception[0]}")
                        continue
                    
                    node = result[0]
                    log(f"위치 {pos} 접근 성공")
                    log(f"노드 타입: {type(node)}")
                    
                    if hasattr(node, "label"):
                        label = node.label()
                        log(f"레이블: {label}")
                    
                    # 자식 노드 정보
                    if hasattr(node, "__len__"):
                        log(f"자식 노드 수: {len(node)}")
                        for i, child in enumerate(node):
                            log(f"자식 {i} 타입: {type(child)}")
                else:
                    log(f"위치 {pos}가 트리 위치에 없음")
            except Exception as e:
                log(f"위치 {pos} 체크 중 예외 발생: {type(e).__name__}: {e}")
                log(traceback.format_exc())
        
        log("디버깅 완료")
        
    except Exception as e:
        log(f"전체 디버깅 과정에서 예외 발생: {type(e).__name__}: {e}")
        log(traceback.format_exc())

# 실행
debug_rst_tree_processing()